# 환경 준비와 실행 점검

로컬 Jupyter 또는 Colab에서 커널·패키지·프로젝트·데이터와 셀 실행을 확인합니다. API 키·모델·client·sandbox는 사용하지 않습니다. 설치 단계에는 패키지 저장소 인터넷 접근이 필요할 수 있지만 이 노트북의 점검 코드는 OpenAI API를 호출하지 않습니다.


<!-- colab-link -->
[독립 실행본 다운로드 안내](../docs/colab.md) · Colab 업로드용


## 준비 목표와 실행 경로

| 단계 | 확인할 내용 | 기대 결과 |
| --- | --- | --- |
| 커널 | Python 버전·실행 파일·작업 폴더 | 사용할 Python과 위치 확인 |
| 패키지 | SDK 버전·모듈 위치 | 현재 커널에서 openai import 가능 |
| 프로젝트 | 공통 helper·프로젝트 루트 | tutorial.json을 기준으로 경로 확인 |
| 데이터 | data/sales.csv | 읽을 수 있는 공통 입력 파일 |
| 로컬 실행 | 작은 계산·assert·출력 | 모든 셀의 기본 동작 확인 |

### 로컬 Jupyter

저장소 루트에서 아래 명령을 터미널에 실행한 뒤 이 노트북을 여세요. VS Code/Jupyter의 커널은 프로젝트 .venv의 Python을 선택합니다. 로컬 재현 기준은 .python-version의 Python 3.12이고 프로젝트 최소 지원 범위는 pyproject.toml의 Python 3.11 이상입니다. 패키지는 uv.lock으로 맞춥니다.

```bash
uv sync --locked
uv run jupyter lab
```

### Colab

[다운로드 안내](../docs/colab.md)의 독립 실행본 setup.ipynb를 노트북 업로드로 여세요. 생성물이 전달된 경우 _build/colab/setup.ipynb가 해당 파일입니다. 준비 셀은 패키지를 직접 설치하고 포함된 helper·데이터를 임시 폴더에 만듭니다. GitHub 주소·저장소 접근·Colab Secrets는 필요하지 않습니다.

Colab은 제공되는 커널 Python을 사용하므로 로컬과 버전이 달라도 됩니다. 아래에서 실제 버전을 확인하세요. 패키지 설치에는 공개 패키지 저장소 접근이 필요합니다. 이 경로는 Docker·사내 네트워크 실습을 대신하지 않습니다. 로컬 원본 bootstrap은 패키지를 설치하지 않으며 독립 실행본 생성 시 설치 코드로 바뀝니다.

아래 셀을 위에서 아래로 실행합니다. uv.lock은 Python 패키지를 고정하지만 운영체제·외부 도구·모델 응답·네트워크와 API 상태까지 고정하지 않습니다.


### 선택 기능과 패키지 설치

기본 설치는 SDK와 Jupyter 실습용 패키지만 포함합니다. 교재 빌드·전체 테스트·관측·SOCKS 프록시는 선택 그룹입니다. 13장 메모리 관측은 `uv run --group telemetry jupyter lab`, Collector 전송은 `uv run --group telemetry-otlp jupyter lab`, SOCKS 프록시 환경은 `uv run --group proxy jupyter lab`을 사용합니다. 여러 그룹은 함께 지정할 수 있습니다. 선택 기능을 사용할 때에는 uv run에도 같은 그룹을 지정하세요. 그룹 없이 다시 실행하면 선택 패키지가 제거될 수 있습니다. VS Code에서 직접 커널을 선택한다면 필요한 그룹을 uv sync로 먼저 설치합니다.

사내 인덱스 환경 변수 때문에 잠금 파일 오류가 나면 `uv sync --locked --default-index https://pypi.org/simple`로 공개 PyPI를 명시해 비교합니다. 다음 실행에도 `uv run --default-index https://pypi.org/simple jupyter lab`처럼 같은 인덱스를 사용합니다. 회사 정책상 사설 인덱스가 필수라면 해당 정책에 맞춘 별도 잠금 구성이 필요합니다. 프록시와 인덱스 URL에 포함될 수 있는 자격 증명을 출력하지 마세요.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


## 1. 현재 Python과 커널 확인

실행 파일이 선택한 커널과 일치하는지 확인하세요. 로컬은 보통 프로젝트 .venv 아래 Python이며 Colab은 런타임의 Python입니다. 경로 모양만으로 실행 환경을 강제 판정하지 않습니다. 프로젝트 최소 버전보다 낮으면 지원되는 커널로 바꾸세요.


In [ ]:
import platform
from pathlib import Path

print("Python:", platform.python_version())
print("Kernel executable:", sys.executable)
print("Working directory:", Path.cwd())
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or newer is required. Select the project .venv or a supported Colab runtime.")


## 2. SDK 설치 확인

아래는 현재 커널의 패키지 버전과 실제 import한 모듈 위치를 확인합니다. SDK가 설치됐다는 사실은 Agents API 권한이나 모델·sandbox 사용 가능 여부를 검증하지 않습니다. 패키지가 없거나 import가 실패하면 아래 문제 해결 표를 확인하세요.


In [ ]:
from importlib.metadata import PackageNotFoundError, version

try:
    sdk_version = version("openai")
    import openai
except (PackageNotFoundError, ImportError) as error:
    raise RuntimeError(
        "OpenAI SDK is unavailable in this kernel. Locally run uv sync --locked and select .venv; "
        "in Colab run the standalone preparation cells."
    ) from error
print("OpenAI SDK version:", sdk_version)
print("OpenAI module location:", openai.__file__)


## 3. 프로젝트·공통 helper·데이터 확인

다음 셀은 공통 helper를 import하고 tutorial.json에서 프로젝트 루트를 찾습니다. data/sales.csv의 경로·읽은 바이트 수를 출력하지만 내용을 분석하지 않습니다. 저장소 안의 하위 폴더에서 실행해도 루트를 찾으며, Colab에서는 준비 셀이 만든 임시 폴더를 사용합니다.

ModuleNotFoundError는 API 키 문제가 아닙니다. 설치·커널 또는 Colab 포함 파일 준비를 확인하세요. 경로나 파일 확인이 실패하면 이후 완료 셀로 넘어가지 말고 해당 준비 단계를 다시 확인합니다.


In [ ]:
try:
    from tutorial_helpers.common import project_root
except ImportError as error:
    raise RuntimeError(
        "Shared helpers are unavailable. Run uv sync --locked and select .venv, "
        "or finish the standalone Colab file-preparation cells."
    ) from error
root = project_root()
if not (root / "tutorial.json").is_file():
    raise FileNotFoundError("Project marker tutorial.json is missing. Open the repository or prepare the Colab workspace.")
csv_path = root / "data/sales.csv"
if not csv_path.is_file():
    raise FileNotFoundError("Shared input data/sales.csv is missing. Check the project root or Colab bundled-file preparation.")
csv_bytes = csv_path.read_bytes()
print("Shared helper:", project_root.__module__)
print("Project root:", root)
print("CSV input:", csv_path)
print("CSV bytes read:", len(csv_bytes))


## 4. 로컬 셀 실행 확인

아래 계산은 현재 Jupyter/Colab Python에서 실행합니다. 같은 숫자 예제를 빠른 시작에서는 원격 sandbox에 요청하므로 두 실행 위치를 비교할 수 있습니다. 여기서 합계가 맞아도 API 인증·모델·sandbox는 아직 확인하지 않은 상태입니다.


In [ ]:
numbers = [10, 20, 30]
summary = {"count": len(numbers), "total": sum(numbers)}
assert summary == {"count": 3, "total": 60}
print(summary)


## 5. 로컬 준비 완료 확인

앞의 셀을 위에서 아래로 모두 실행한 뒤 아래 완료 요약을 읽으세요. 현재 kernel·SDK·프로젝트·데이터와 계산 출력만 확인했으며 API client나 원격 자원은 만들지 않았습니다. 이 노트북에는 실행 허용 스위치가 필요 없습니다.


In [ ]:
print("Local preparation complete:", {
    "python": platform.python_version(),
    "openai_sdk": sdk_version,
    "project_root": str(root),
    "input_bytes": len(csv_bytes),
    "local_result": summary,
})
print("API authentication, model access, and sandbox execution have not been tested.")


## 문제 해결과 다음 단계

| 증상 | 다음 행동 |
| --- | --- |
| openai 패키지 없음·import 실패 | 로컬 uv sync --locked 후 .venv 커널 확인, Colab은 설치 셀 완료 확인 |
| tutorial_helpers import 실패 | 프로젝트 설치·작업 위치·커널 또는 Colab 포함 파일 준비 확인 |
| tutorial.json 또는 CSV 없음 | 프로젝트 루트·저장소 파일·Colab 파일 준비 확인 |
| 설치 후에도 이전 import 오류 지속 | 현재 실행 파일 확인 후 필요하면 커널/런타임 재시작, 준비 셀부터 재실행 |
| Python 버전이 로컬 기준과 다름 | 프로젝트 최소 버전과 현재 커널 확인, Colab은 로컬과 동일 버전일 필요 없음 |

설치가 실패했는데 계속 실행하거나 키·토큰 값을 출력해서 해결하지 않습니다. 이 노트북의 성공은 로컬 준비 완료이며 API 인증 성공과 다릅니다.

- 키 없이 개념 학습: [1장 핵심 개념](chapters/01_concepts.ipynb)으로 이동하세요.
- 실제 API 실습: [빠른 시작](quickstart.ipynb)에서 키·모델 설정과 명시적인 실행 허용을 진행하세요. 로컬은 Git 제외 .env.local, Colab은 Secrets를 사용하며 키를 셀·출력·sandbox에 넣지 않습니다.

### 선택: 웹 교재 미리보기

웹 미리보기는 Python 노트북 실행과 별개입니다. 배포본과 같은 한영 페이지·다운로드 구조를 로컬에서 확인하려면 아래 명령을 터미널에서 실행하세요. 빌드 도구·Node.js 준비 등은 저장소 README의 로컬 교재 미리보기 안내를 참고하세요.

```bash
uv run --group docs python scripts/preview_book.py --build
```

미리보기 웹페이지는 교재 열람용이며 Python 실행은 Jupyter/Colab에서 수행합니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Quickstart — Prerequisites](https://developers.openai.com/api/docs/guides/agents-api/quickstart#prerequisites) — Python SDK와 API 키·권한 준비.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
